# put the paths and variables for roboflow here

In [30]:
#ebi full model

ROBOFLOW_API_KEY="AgfWlgODe2lXi5Nh5VTu"
ROBOFLOW_WORKSPACE="made-srcnb"
ROBOFLOW_PROJECT="ebi-fused"
DATASET_VERSION=3
DATASET_TYPE="multiclass"
DATASET_PARENT_DIR="/mnt/EBI-SHARE/06_Data/labelstudio/datasets"


destination_path=f"{DATASET_PARENT_DIR}/{DATASET_TYPE}/{ROBOFLOW_PROJECT}-{DATASET_VERSION}"

In [8]:
#windows

ROBOFLOW_API_KEY="AgfWlgODe2lXi5Nh5VTu"
ROBOFLOW_WORKSPACE="tafeln"
ROBOFLOW_PROJECT="fensterfurmodel"
DATASET_VERSION=5
DATASET_TYPE="windows"
DATASET_PARENT_DIR="/mnt/EBI-SHARE/06_Data/labelstudio/datasets"


destination_path=f"{DATASET_PARENT_DIR}/{DATASET_TYPE}/{ROBOFLOW_PROJECT}-{DATASET_VERSION}"

In [2]:
#lights

ROBOFLOW_API_KEY="AgfWlgODe2lXi5Nh5VTu"
ROBOFLOW_WORKSPACE="tree-detection-zocbs"
ROBOFLOW_PROJECT="light_detect_3"
DATASET_VERSION=2
DATASET_TYPE="lights"
DATASET_PARENT_DIR="/mnt/EBI-SHARE/06_Data/labelstudio/datasets"

destination_path=f"{DATASET_PARENT_DIR}/{DATASET_TYPE}/{ROBOFLOW_PROJECT}-{DATASET_VERSION}"

In [31]:
import shutil
import subprocess
from roboflow import Roboflow


rf = Roboflow(api_key=ROBOFLOW_API_KEY)
print("key accepted")
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
print(f"project {ROBOFLOW_PROJECT} found in workspace {ROBOFLOW_WORKSPACE}")
version = project.version(DATASET_VERSION)
print(f"version {DATASET_VERSION} found in project {ROBOFLOW_PROJECT}")
dataset = version.download("yolov8")



shutil.copytree(dataset.location, destination_path, dirs_exist_ok=True)
shutil.rmtree(dataset.location)

subprocess.run(["chmod", "-R", "777", f"{destination_path}"])

print(f"dataset copied to {destination_path} and permissions set to 777")

key accepted
loading Roboflow workspace...
loading Roboflow project...
project ebi-fused found in workspace made-srcnb
version 3 found in project ebi-fused
Exporting format yolov8 in progress : 85.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to ebi-fused-3 in yolov8:: 100%|██████████| 6808/6808 [00:00<00:00, 11804.64it/s]


dataset copied to /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-3 and permissions set to 777


# convert to label studio

In [28]:
import yaml
import os

yaml_path = f"{destination_path}/data.yaml"
with open(yaml_path) as f:
    classes = yaml.safe_load(f).get("names", [])
    

for subfolder in os.listdir(destination_path):
    subfolder_path = os.path.join(destination_path, subfolder)
    if os.path.isdir(subfolder_path):
        classes_txt = os.path.join(subfolder_path, "classes.txt")
        with open(classes_txt, "w") as f:
            f.write("\n".join(classes))

In [29]:
import os
import subprocess

for subfolder in os.listdir(destination_path):
    subfolder_path = os.path.join(destination_path, subfolder)
    if os.path.isdir(subfolder_path):
        try:
            subprocess.run([
                "label-studio-converter", "import", "yolo",
                "-i", subfolder_path,
                "-o", f"{subfolder_path}/label-studio.json",
                "--image-root-url", f"{subfolder_path}/images"
            ])
        except Exception as e:
            print(f"Error processing {subfolder_path}: {e}")

INFO:root:Reading YOLO notes and categories from /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-2/train
INFO:root:Found 3 categories
INFO:root:Converting labels from /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-2/train/labels
INFO:root:image extensions->, ['.jpg']
INFO:root:Saving Label Studio JSON to /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-2/train/label-studio.json



  1. Create a new project in Label Studio
  2. Use Labeling Config from "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-2/train/label-studio.label_config.xml"
  3. Setup serving for images
       E.g. you can use Local Storage (or others):
       https://labelstud.io/guide/storage.html#Local-storage
       See tutorial here:
https://github.com/HumanSignal/label-studio-converter/tree/master?tab=readme-ov-file#yolo-to-label-studio-converter
       
  4. Import "/mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-2/train/label-studio.json" to the project



In [13]:
print(destination_path)

/mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-1


In [24]:
!label-studio-converter import yolo \
  -i /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-1/train \
  -o ls-tasks.json \
  --image-root-url /data/local-files/?d=datasets/multiclass/ebi-fused-1/train/images

INFO:root:Reading YOLO notes and categories from /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-1/train
INFO:root:Found 3 categories
INFO:root:Converting labels from /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-fused-1/train/labels
INFO:root:image extensions->, ['.jpg']
INFO:root:Saving Label Studio JSON to /home/freeze/GIT/onemanstreasure/notebooks/ls-tasks.json

  1. Create a new project in Label Studio
  2. Use Labeling Config from "/home/freeze/GIT/onemanstreasure/notebooks/ls-tasks.label_config.xml"
  3. Setup serving for images
       E.g. you can use Local Storage (or others):
       https://labelstud.io/guide/storage.html#Local-storage
       See tutorial here:
https://github.com/HumanSignal/label-studio-converter/tree/master?tab=readme-ov-file#yolo-to-label-studio-converter
       
  4. Import "/home/freeze/GIT/onemanstreasure/notebooks/ls-tasks.json" to the project



In [ ]:
http://localhost:8081/data/local-files/?d=datasets/multiclass/ebi-fused-1/train/images/-47104722_jpg.rf.04ee0f9cbb75ffdfadc14e117275c20e.jpg